# M5: 20 or 50 products, two training years, one Random Forest

Run in **Google Colab → CPU → Runtime → Run all**. Self-contained: no repository clone, credentials, Drive mount, or Store database. This downloads the public 116 MB M5 Parquet file, filters CA_1 / FOODS, samples 20 products using training data only (seed 42), and runs the exact bundled RetailMind forecasting functions.

Two full calendar training years precede 84 held-out days: two 28-day validation blocks and one final 28-day test. Expanding training windows may use observations from earlier blocks, never from the forecast block. The Random Forest recipe is fixed before evaluation: 15 trees, depth 8, seed 42; direct daily predictions to day 28 with no recursive actuals. Validation selects only the any-sale-day threshold. Do not tune against the final test.

**Research only.** Source sales do not establish lost demand, stock availability, or local Store accuracy. Original dates and anonymous product IDs stay unchanged. No inventory is modified. [Dataset](https://huggingface.co/datasets/skforecast/m5_daily), [shared design](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).

Set PRODUCT_COUNT to 20 or 50 in the run cell. The 50-product run retains the original 20 and adds 30 training-eligible products with seed 42. Compare both runs on the original 20; all-50 averages describe a different population. The previously inspected final window is development evidence, not independent confirmation.


In [ ]:
# Pin experiment packages. If Colab requests a restart, restart then Run all.
%pip install -q numpy==2.4.4 pandas==2.3.3 scikit-learn==1.9.1 pyarrow==26.0.0 matplotlib==3.11.2


## Frozen shared source
These four files are embedded verbatim from this experiment branch so the notebook works without access to the private repository. Update the bundle if shared source changes; the repository test checks this. Only forecasting code is included, no operational data or configuration.


In [ ]:
BUNDLED_SOURCES = {
  "ml_cv.py": "import numpy as np\nimport pandas as pd\nfrom typing import List, Tuple\n\ndef rolling_origin_split(df: pd.DataFrame, date_col: str, horizon_days: int, n_splits: int = 3) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:\n    \"\"\"\n    Creates rolling-origin chronological splits preserving calendar horizons (#365).\n    df must have date_col.\n    Returns list of (train_df, test_df) in chronological order.\n    \"\"\"\n    df = df.sort_values(by=date_col)\n    max_date = pd.to_datetime(df[date_col].max())\n    min_date = pd.to_datetime(df[date_col].min())\n    \n    splits = []\n    for i in range(n_splits):\n        test_end_date = max_date - pd.Timedelta(days=i * horizon_days)\n        test_start_date = test_end_date - pd.Timedelta(days=horizon_days - 1)\n        origin_date = test_start_date - pd.Timedelta(days=1)\n        \n        if origin_date < min_date:\n            break\n            \n        test_mask = (df[date_col] >= test_start_date) & (df[date_col] <= test_end_date)\n        train_mask = df[date_col] <= origin_date\n        \n        if not train_mask.any() or not test_mask.any():\n            break\n            \n        splits.append((df[train_mask], df[test_mask]))\n        \n    return splits[::-1]  # Return in chronological order\n",
  "ml_evaluation.py": "import numpy as np\n\ndef calculate_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict:\n    actual = np.asarray(actual, dtype=float)\n    predicted = np.asarray(predicted, dtype=float)\n    \n    population = int(len(actual))\n    if population == 0:\n        return {\n            'mae': None,\n            'rmse': None,\n            'wape': None,\n            'population': 0,\n            'underprediction_count': 0,\n            'overprediction_count': 0,\n            'exact_count': 0,\n            'underprediction_units': 0.0,\n            'overprediction_units': 0.0,\n            'missing_metric_reason': \"Insufficient coverage\"\n        }\n        \n    diff = predicted - actual\n    abs_diff = np.abs(diff)\n    \n    mae = float(np.mean(abs_diff))\n    rmse = float(np.sqrt(np.mean(diff ** 2)))\n    total_actual = float(np.sum(actual))\n    \n    under_mask = predicted < actual\n    over_mask = predicted > actual\n    exact_mask = predicted == actual\n    \n    underprediction_count = int(np.sum(under_mask))\n    overprediction_count = int(np.sum(over_mask))\n    exact_count = int(np.sum(exact_mask))\n    \n    underprediction_units = float(np.sum(np.maximum(0.0, actual - predicted)))\n    overprediction_units = float(np.sum(np.maximum(0.0, predicted - actual)))\n    \n    if total_actual == 0:\n        wape = None\n        missing_metric_reason = \"Zero actual demand\"\n    elif not np.isfinite(total_actual) or total_actual < 0:\n        wape = None\n        missing_metric_reason = \"Invalid actual demand total\"\n    else:\n        wape = float(np.sum(abs_diff) / total_actual)\n        missing_metric_reason = None\n        \n    return {\n        'mae': mae,\n        'rmse': rmse,\n        'wape': wape,\n        'population': population,\n        'underprediction_count': underprediction_count,\n        'overprediction_count': overprediction_count,\n        'exact_count': exact_count,\n        'underprediction_units': underprediction_units,\n        'overprediction_units': overprediction_units,\n        'missing_metric_reason': missing_metric_reason\n    }\n\ndef seasonal_naive_forecast(history: np.ndarray, horizon: int, season_length: int = 7) -> np.ndarray:\n    \"\"\"\n    Generate a seasonal naive forecast.\n    If there is enough history, use the same day from the previous season.\n    If not, use the last available value.\n    \"\"\"\n    forecast = np.zeros(horizon)\n    for i in range(horizon):\n        steps_back = season_length - (i % season_length)\n        if len(history) >= steps_back:\n            forecast[i] = history[-steps_back]\n        elif len(history) > 0:\n            forecast[i] = history[-1]\n        else:\n            forecast[i] = 0.0\n            \n    return forecast\n",
  "feature_pipeline.py": "\"\"\"One cutoff-aware feature contract for observed daily demand (#365, #366).\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder\n\nfrom ml_cv import rolling_origin_split\nfrom ml_evaluation import calculate_metrics, seasonal_naive_forecast\n\nFEATURE_VERSION = \"2.0.0\"\nFEATURE_COLUMNS = [\n    \"product_id\", \"horizon_day\", \"lag_1\", \"lag_7\", \"lag_14\", \"lag_28\",\n    \"rolling_mean_7\", \"rolling_std_7\", \"rolling_mean_28\", \"rolling_std_28\",\n    \"day_of_week\", \"day_of_month\", \"month\", \"is_weekend\",\n]\nMODEL_PARAMETERS = {\"n_estimators\": 15, \"max_depth\": 8, \"random_state\": 42}\n\n\ndef normalize_history(df):\n    if \"Store ID\" in df and df[\"Store ID\"].nunique() != 1:\n        raise ValueError(\"Select exactly one isolated dataset Store\")\n    renamed = df.rename(columns={\n        \"Date\": \"date\", \"Product ID\": \"product_id\", \"Units Sold\": \"units_sold\",\n    })\n    columns = [\"date\", \"product_id\", \"units_sold\"]\n    versioned = \"available_date\" in renamed\n    if versioned:\n        columns.append(\"available_date\")\n    for extra in [\"price\", \"is_on_promotion\", \"out_of_stock\", \"classification\", \"source\", \"recorded_sales_units\"]:\n        if extra in renamed:\n            columns.append(extra)\n    data = renamed[columns].copy()\n    data[\"date\"] = pd.to_datetime(data[\"date\"], errors=\"raise\")\n    data[\"units_sold\"] = pd.to_numeric(data[\"units_sold\"], errors=\"coerce\")\n    if data.empty or data[[\"date\", \"product_id\"]].isna().any().any():\n        raise ValueError(\"No usable demand history or missing observations\")\n    if (data.date != data.date.dt.normalize()).any():\n        raise ValueError(\"Demand history must use daily dates\")\n    values = data.units_sold.dropna()\n    if not np.isfinite(values).all() or (values < 0).any():\n        raise ValueError(\"Demand observations must be finite and nonnegative\")\n    keys = [\"product_id\", \"date\"]\n    if versioned:\n        data[\"available_date\"] = pd.to_datetime(data.available_date, errors=\"raise\")\n        if (data.available_date.isna().any() or (data.available_date < data.date).any()\n                or (data.available_date != data.available_date.dt.normalize()).any()):\n            raise ValueError(\"Invalid observation availability date\")\n        keys.append(\"available_date\")\n    if data.duplicated(keys).any():\n        raise ValueError(\"Duplicate daily demand observations\")\n    return data.sort_values(keys).reset_index(drop=True)\n\n\ndef history_at(df, cutoff_date):\n    \"\"\"Latest known daily value; null revisions withdraw unusable observations.\"\"\"\n    data = normalize_history(df)\n    cutoff = pd.Timestamp(cutoff_date)\n    data = data[data.date <= cutoff]\n    if \"available_date\" in data:\n        data = data[data.available_date <= cutoff].drop_duplicates([\"product_id\", \"date\"], keep=\"last\")\n        data = data.dropna(subset=[\"units_sold\"]).drop(columns=\"available_date\")\n    else:\n        data = data.dropna(subset=[\"units_sold\"])\n    if data.empty:\n        raise ValueError(\"No usable demand history at cutoff\")\n    return data\n\n\ndef build_feature_matrix(df, cutoff_date, horizon_days=28):\n    \"\"\"\n    Construct supported dated feature windows (#365, #366).\n    Lag 1 is the observed origin day; preserves calendar dates without inventing zero demand.\n    Skips unsupported product/origin combinations and records reasons.\n    \"\"\"\n    data = history_at(df, cutoff_date)\n    cutoff = pd.Timestamp(cutoff_date)\n    expected = pd.date_range(cutoff - pd.Timedelta(days=27), cutoff)\n    rows = []\n    exclusion_reasons = {}\n\n    for pid, group in data.groupby(\"product_id\", sort=True):\n        window = group[group.date.isin(expected)]\n        sales_by_date = dict(zip(window.date, window.units_sold))\n\n        # Check origin day observation (lag 1)\n        if cutoff not in sales_by_date:\n            exclusion_reasons[pid] = f\"Product {pid} requires origin observation at {cutoff.date()}\"\n            continue\n\n        # Check if window has unsupported gaps (#365)\n        # Operating mode: if product has regular weekend sales in history, weekends are operating days\n        has_classification = 'classification' in group.columns\n        weekend_operating = (group.date.dt.dayofweek >= 5).any() and not has_classification\n        has_unsupported_gap = False\n        for d in expected:\n            if d not in sales_by_date:\n                if weekend_operating:\n                    has_unsupported_gap = True\n                    break\n                elif d.dayofweek < 5:\n                    has_unsupported_gap = True\n                    break\n\n        if has_unsupported_gap:\n            exclusion_reasons[pid] = f\"Product {pid} requires 28 consecutive observed days or supported closures at {cutoff.date()}\"\n            continue\n\n        # Calculate lag dates:\n        # lag_1: cutoff\n        # lag_7: cutoff - 6 days\n        # lag_14: cutoff - 13 days\n        # lag_28: cutoff - 27 days\n        lag_dates = {\n            1: cutoff,\n            7: cutoff - pd.Timedelta(days=6),\n            14: cutoff - pd.Timedelta(days=13),\n            28: cutoff - pd.Timedelta(days=27),\n        }\n\n        # Resolve lag observations preserving calendar dates and supported closures (#365)\n        lags = {}\n        lags_valid = True\n        group_history_sales = dict(zip(group.loc[group.date <= cutoff, \"date\"], group.loc[group.date <= cutoff, \"units_sold\"]))\n        sorted_history_dates = sorted(group_history_sales.keys())\n        for lag, ldate in lag_dates.items():\n            if ldate in sales_by_date:\n                lags[f\"lag_{lag}\"] = float(sales_by_date[ldate])\n            else:\n                # If date falls on a supported closure, resolve to latest observed trading day at or before ldate\n                prev_dates = [d for d in sorted_history_dates if d <= ldate]\n                if prev_dates:\n                    lags[f\"lag_{lag}\"] = float(group_history_sales[prev_dates[-1]])\n                elif len(window) == 28:\n                    sales_arr = window.units_sold.to_numpy(dtype=float)\n                    lags[f\"lag_{lag}\"] = float(sales_arr[-lag])\n                else:\n                    lags_valid = False\n                    break\n\n        if not lags_valid:\n            exclusion_reasons[pid] = f\"Product {pid} missing lag observations at {cutoff.date()}\"\n            continue\n\n        # Rolling summaries: computed over observed usable days in calendar window without inventing zero demand\n        w7_dates = pd.date_range(cutoff - pd.Timedelta(days=6), cutoff)\n        s7 = [float(sales_by_date[d]) for d in w7_dates if d in sales_by_date]\n        s28 = [float(sales_by_date[d]) for d in expected if d in sales_by_date]\n\n        rolling = {\n            \"rolling_mean_7\": float(np.mean(s7)),\n            \"rolling_std_7\": float(np.std(s7)),\n            \"rolling_mean_28\": float(np.mean(s28)),\n            \"rolling_std_28\": float(np.std(s28)),\n        }\n\n        for h in range(1, horizon_days + 1):\n            target = cutoff + pd.Timedelta(days=h)\n            rows.append({\n                \"product_id\": pid, \"origin_date\": cutoff, \"target_date\": target,\n                \"horizon_day\": h, **lags, **rolling,\n                \"day_of_week\": target.dayofweek, \"day_of_month\": target.day,\n                \"month\": target.month, \"is_weekend\": int(target.dayofweek >= 5),\n            })\n\n    if not rows:\n        reasons_msg = \"; \".join(exclusion_reasons.values()) if exclusion_reasons else f\"No active products have supported feature windows at {cutoff.date()}\"\n        raise ValueError(f\"Product requires 28 consecutive observed days at {cutoff.date()}: {reasons_msg}\")\n\n    return pd.DataFrame(rows)\n\n\ndef extract_training_targets(df, cutoff_date, horizon_days=28, label_cutoff=None):\n    data = history_at(df, label_cutoff or pd.Timestamp(cutoff_date) + pd.Timedelta(days=horizon_days))\n    features = build_feature_matrix(df, cutoff_date, horizon_days)\n    targets = features.merge(data, left_on=[\"product_id\", \"target_date\"],\n                             right_on=[\"product_id\", \"date\"], how=\"left\", validate=\"one_to_one\")\n    if targets.units_sold.isna().any():\n        raise ValueError(\"Missing observed training targets\")\n    return targets.units_sold\n\n\ndef build_training_rows(df, cutoff_date):\n    \"\"\"Weekly origins, with all observed labels up to training cutoff (#365).\"\"\"\n    data = history_at(df, cutoff_date)\n    cutoff = pd.Timestamp(cutoff_date)\n    data = data[data.date <= cutoff]\n    if data.empty:\n        raise ValueError(\"No usable demand history\")\n\n    first_origin = data.groupby(\"product_id\").date.min().max() + pd.Timedelta(days=27)\n    last_origin = min(cutoff, data.date.max()) - pd.Timedelta(days=28)\n    rows = []\n\n    # Attempt origins along 7D intervals\n    for origin in pd.date_range(first_origin, last_origin, freq=\"7D\"):\n        try:\n            features = build_feature_matrix(df, origin)\n        except ValueError:\n            continue\n        # Only observed target dates become training rows: excluded/closed days do not become targets\n        merged = features.merge(data[[\"product_id\", \"date\", \"units_sold\"]],\n                                left_on=[\"product_id\", \"target_date\"],\n                                right_on=[\"product_id\", \"date\"], how=\"inner\")\n        if not merged.empty:\n            rows.append(merged.drop(columns=[\"date\"]))\n\n    # If first_origin fell on a closed day, try offsetting to open days\n    if not rows:\n        for offset in range(1, 7):\n            alt_first = first_origin + pd.Timedelta(days=offset)\n            if alt_first > last_origin:\n                break\n            for origin in pd.date_range(alt_first, last_origin, freq=\"7D\"):\n                try:\n                    features = build_feature_matrix(df, origin)\n                except ValueError:\n                    continue\n                merged = features.merge(data[[\"product_id\", \"date\", \"units_sold\"]],\n                                        left_on=[\"product_id\", \"target_date\"],\n                                        right_on=[\"product_id\", \"date\"], how=\"inner\")\n                if not merged.empty:\n                    rows.append(merged.drop(columns=[\"date\"]))\n            if rows:\n                break\n\n    if not rows:\n        raise ValueError(\"At least 56 consecutive observed days are required for training\")\n\n    return pd.concat(rows, ignore_index=True)\n\n\ndef _fit_model(rows):\n    model = make_pipeline(\n        ColumnTransformer([('product', OneHotEncoder(), ['product_id'])], remainder='passthrough'),\n        RandomForestRegressor(**MODEL_PARAMETERS),\n    )\n    return model.fit(rows[FEATURE_COLUMNS], rows.units_sold)\n\n\ndef train_and_evaluate(df):\n    \"\"\"\n    Three rolling 28-day holdouts; score per-product 7/14/28-day totals (#365, #366).\n    Evaluates complete usable Product/horizon totals after worker's nonnegative daily rounding.\n    Applies identical explicit closure treatment to candidate and baseline.\n    Reports included/excluded products, fractional diagnostics, and under/overprediction.\n    \"\"\"\n    history = normalize_history(df)\n    data = history_at(history, history.date.max())\n    training = build_training_rows(history, data.date.max())\n    folds = []\n    totals = {\n        h: {\n            \"actual\": [], \"predicted\": [], \"baseline\": [],\n            \"predicted_fractional\": [], \"baseline_fractional\": []\n        }\n        for h in (7, 14, 28)\n    }\n\n    splits = rolling_origin_split(data, \"date\", 28, n_splits=3)\n    if len(splits) != 3:\n        raise ValueError(\"Insufficient history for three chronological holdouts\")\n\n    all_products = [int(p) for p in sorted(data.product_id.unique())]\n    product_holdout_complete = {pid: True for pid in all_products}\n    product_exclusion_reasons = {}\n\n    for past, held_out in splits:\n        test_start = held_out.date.min()\n        test_end = held_out.date.max()\n        origin = test_start - pd.Timedelta(days=1)\n        if origin not in past.date.values:\n            recent = past[past.date <= origin]\n            if not recent.empty:\n                origin = recent.date.max()\n\n        fit_rows = (build_training_rows(history, origin) if \"available_date\" in history else\n                    training[training.origin_date + pd.Timedelta(days=28) <= origin])\n        if fit_rows.empty:\n            raise ValueError(\"Insufficient history for three chronological holdouts\")\n\n        model = _fit_model(fit_rows)\n        try:\n            features = build_feature_matrix(history, origin)\n        except ValueError:\n            raise ValueError(\"Insufficient history for three chronological holdouts\")\n\n        past_at_origin = history_at(history, origin)\n        fold_products_included = []\n        fold_products_excluded = []\n        fold_evaluations = {}\n\n        fold_actual = {7: [], 14: [], 28: []}\n        fold_predicted = {7: [], 14: [], 28: []}\n        fold_baseline = {7: [], 14: [], 28: []}\n        fold_predicted_frac = {7: [], 14: [], 28: []}\n        fold_baseline_frac = {7: [], 14: [], 28: []}\n\n        for pid, pfeatures in features.groupby(\"product_id\"):\n            pid = int(pid)\n            p_history = history[history.product_id == pid]\n            has_classification = \"classification\" in p_history.columns\n            weekend_operating = (p_history.date.dt.dayofweek >= 5).any() and not has_classification\n\n            # Check holdout completeness across all 28 calendar days (#366)\n            holdout_valid = True\n            holdout_actual = {}\n            is_recorded_closure = {}\n\n            for h in range(1, 29):\n                tdate = origin + pd.Timedelta(days=h)\n                match = p_history[p_history.date == tdate]\n\n                if not match.empty:\n                    row = match.iloc[-1]\n                    if has_classification:\n                        cls = str(row.get(\"classification\", \"\"))\n                        if cls == \"closed\":\n                            is_recorded_closure[h] = True\n                            holdout_actual[h] = 0.0\n                        elif cls in (\"valid_demand\", \"genuine_zero\"):\n                            is_recorded_closure[h] = False\n                            units = row.get(\"units_sold\")\n                            holdout_actual[h] = float(units) if pd.notna(units) else 0.0\n                        else:  # stockout_censored or missing_evidence\n                            holdout_valid = False\n                            break\n                    else:\n                        units = row.get(\"units_sold\")\n                        if pd.notna(units):\n                            is_recorded_closure[h] = False\n                            holdout_actual[h] = float(units)\n                        elif tdate.dayofweek >= 5 and not weekend_operating:\n                            is_recorded_closure[h] = True\n                            holdout_actual[h] = 0.0\n                        else:\n                            holdout_valid = False\n                            break\n                else:\n                    if tdate.dayofweek >= 5 and not weekend_operating:\n                        is_recorded_closure[h] = True\n                        holdout_actual[h] = 0.0\n                    else:\n                        holdout_valid = False\n                        break\n\n            if not holdout_valid:\n                product_holdout_complete[pid] = False\n                reason = f\"Missing trading evidence or stockout during holdout at {origin.date()}\"\n                product_exclusion_reasons[pid] = reason\n                fold_products_excluded.append(pid)\n                continue\n\n            fold_products_included.append(pid)\n            preds = model.predict(pfeatures[FEATURE_COLUMNS])\n            pred_map = dict(zip(pfeatures.horizon_day, preds))\n\n            # Weekly Seasonal Naive from permitted past 28 calendar days\n            w_dates = pd.date_range(origin - pd.Timedelta(days=27), origin)\n            p_past = past_at_origin[past_at_origin.product_id == pid]\n            past_sales_by_date = dict(zip(p_past.date, p_past.units_sold))\n            past_28_units = np.array([float(past_sales_by_date.get(d, 0.0)) for d in w_dates], dtype=float)\n            base_forecast = seasonal_naive_forecast(past_28_units, 28, season_length=7)\n            base_map = dict(zip(range(1, 29), base_forecast))\n\n            # Daily predictions with nonnegative integer rounding & explicit closure treatment (#366)\n            pred_daily_map = {}\n            base_daily_map = {}\n            pred_frac_map = {}\n            base_frac_map = {}\n\n            for h in range(1, 29):\n                if is_recorded_closure.get(h, False):\n                    # Identical explicit closure treatment: 0 demand when Store is closed\n                    pred_daily_map[h] = 0\n                    base_daily_map[h] = 0\n                    pred_frac_map[h] = 0.0\n                    base_frac_map[h] = 0.0\n                else:\n                    p_val = float(pred_map.get(h, 0.0))\n                    b_val = float(base_map.get(h, 0.0))\n                    pred_daily_map[h] = max(0, int(round(p_val)))\n                    base_daily_map[h] = max(0, int(round(b_val)))\n                    pred_frac_map[h] = p_val\n                    base_frac_map[h] = b_val\n\n            for horizon in (7, 14, 28):\n                a = sum(holdout_actual[h] for h in range(1, horizon + 1))\n                p = sum(pred_daily_map[h] for h in range(1, horizon + 1))\n                b = sum(base_daily_map[h] for h in range(1, horizon + 1))\n                p_frac = sum(pred_frac_map[h] for h in range(1, horizon + 1))\n                b_frac = sum(base_frac_map[h] for h in range(1, horizon + 1))\n\n                fold_actual[horizon].append(a)\n                fold_predicted[horizon].append(p)\n                fold_baseline[horizon].append(b)\n                fold_predicted_frac[horizon].append(p_frac)\n                fold_baseline_frac[horizon].append(b_frac)\n\n                totals[horizon][\"actual\"].append(a)\n                totals[horizon][\"predicted\"].append(p)\n                totals[horizon][\"baseline\"].append(b)\n                totals[horizon][\"predicted_fractional\"].append(p_frac)\n                totals[horizon][\"baseline_fractional\"].append(b_frac)\n\n        if not fold_products_included:\n            raise ValueError(\"Insufficient holdout coverage for candidate evaluation\")\n\n        for horizon in (7, 14, 28):\n            a_arr = np.array(fold_actual[horizon], dtype=float)\n            p_arr = np.array(fold_predicted[horizon], dtype=float)\n            b_arr = np.array(fold_baseline[horizon], dtype=float)\n            p_frac_arr = np.array(fold_predicted_frac[horizon], dtype=float)\n            b_frac_arr = np.array(fold_baseline_frac[horizon], dtype=float)\n\n            m = calculate_metrics(a_arr, p_arr)\n            m.update({f\"baseline_{k}\": v for k, v in calculate_metrics(a_arr, b_arr).items()})\n            frac_m = calculate_metrics(a_arr, p_frac_arr)\n            frac_b = calculate_metrics(a_arr, b_frac_arr)\n            m[\"mae_fractional\"] = frac_m[\"mae\"]\n            m[\"rmse_fractional\"] = frac_m[\"rmse\"]\n            m[\"wape_fractional\"] = frac_m[\"wape\"]\n            m[\"baseline_mae_fractional\"] = frac_b[\"mae\"]\n            m[\"baseline_rmse_fractional\"] = frac_b[\"rmse\"]\n            m[\"baseline_wape_fractional\"] = frac_b[\"wape\"]\n            m[\"coverage\"] = len(fold_products_included)\n            fold_evaluations[str(horizon)] = m\n\n        folds.append({\n            \"origin\": str(origin.date()),\n            \"training_label_end\": str(fit_rows.target_date.max().date()),\n            \"training_rows\": len(fit_rows),\n            \"validation_start\": str((origin + pd.Timedelta(days=1)).date()),\n            \"validation_end\": str((origin + pd.Timedelta(days=28)).date()),\n            \"included_products\": fold_products_included,\n            \"excluded_products\": fold_products_excluded,\n            \"evaluations\": fold_evaluations,\n        })\n\n    evaluations = {}\n    eligible_pids = [int(p) for p in all_products if product_holdout_complete.get(p, False)]\n    for horizon, values in totals.items():\n        actual = np.array(values[\"actual\"], dtype=float)\n        predicted = np.array(values[\"predicted\"], dtype=float)\n        baseline = np.array(values[\"baseline\"], dtype=float)\n        pred_frac = np.array(values[\"predicted_fractional\"], dtype=float)\n        base_frac = np.array(values[\"baseline_fractional\"], dtype=float)\n\n        metrics = calculate_metrics(actual, predicted)\n        base_metrics = calculate_metrics(actual, baseline)\n        metrics.update({f\"baseline_{k}\": v for k, v in base_metrics.items()})\n\n        frac_m = calculate_metrics(actual, pred_frac)\n        frac_b = calculate_metrics(actual, base_frac)\n        metrics[\"mae_fractional\"] = frac_m[\"mae\"]\n        metrics[\"rmse_fractional\"] = frac_m[\"rmse\"]\n        metrics[\"wape_fractional\"] = frac_m[\"wape\"]\n        metrics[\"baseline_mae_fractional\"] = frac_b[\"mae\"]\n        metrics[\"baseline_rmse_fractional\"] = frac_b[\"rmse\"]\n        metrics[\"baseline_wape_fractional\"] = frac_b[\"wape\"]\n\n        metrics[\"coverage\"] = len(eligible_pids)\n        metrics[\"population\"] = int(len(actual))\n        evaluations[horizon] = metrics\n\n    model = _fit_model(training)\n\n    return {\n        \"model\": model,\n        \"evaluations\": evaluations,\n        \"metadata\": {\n            \"feature_version\": FEATURE_VERSION, \"features\": FEATURE_COLUMNS,\n            \"parameters\": MODEL_PARAMETERS, \"training_rows\": len(training),\n            \"training_window_start\": str((training.origin_date.min() - pd.Timedelta(days=27)).date()),\n            \"training_window_end\": str(training.target_date.max().date()),\n            \"data_window_start\": str(data.date.min().date()),\n            \"data_window_end\": str(data.date.max().date()),\n            \"products\": eligible_pids if eligible_pids else all_products,\n            \"included_products\": eligible_pids,\n            \"excluded_products\": [int(pid) for pid in all_products if pid not in eligible_pids],\n            \"exclusion_reasons\": {int(k): v for k, v in product_exclusion_reasons.items()},\n            \"observations\": len(data), \"folds\": folds,\n        },\n    }\n",
  "m5_sales_experiment.py": "\"\"\"Offline M5 experiment; never imports public sales into the Store database.\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\n\nfrom feature_pipeline import (\n    FEATURE_COLUMNS, FEATURE_VERSION, MODEL_PARAMETERS, _fit_model,\n    build_feature_matrix, build_training_rows, extract_training_targets, normalize_history,\n)\nfrom ml_cv import rolling_origin_split\nfrom ml_evaluation import calculate_metrics, seasonal_naive_forecast\n\nSOURCE_REVISION = \"9eee585c46882aa28aaaf43733a5c55c3e32f058\"\nSOURCE_URL = f\"https://huggingface.co/datasets/skforecast/m5_daily/resolve/{SOURCE_REVISION}/m5_daily.parquet\"\n\n\ndef prepare_history(raw, product_count=20):\n    \"\"\"Two calendar years before three 28-day holdouts; sample using past only.\"\"\"\n    data = raw.loc[(raw.store_id == \"CA_1\") & (raw.cat_id == \"FOODS\"),\n                   [\"datetime\", \"item_id\", \"sales\"]].rename(columns={\n                       \"datetime\": \"date\", \"item_id\": \"product_id\", \"sales\": \"units_sold\"})\n    data[\"product_id\"] = data.product_id.astype(\"string\")\n    data = normalize_history(data)\n    if (data.units_sold % 1 != 0).any():\n        raise ValueError(\"M5 sales must be whole units\")\n    end = data.date.max()\n    training_end = end - pd.Timedelta(days=84)\n    start = training_end - pd.DateOffset(years=2) + pd.Timedelta(days=1)\n    past = data[data.date.between(start, training_end)]\n    days = (training_end - start).days + 1\n    eligible = past.groupby(\"product_id\").agg(days=(\"date\", \"size\"), units=(\"units_sold\", \"sum\"))\n    codes = eligible.index[(eligible.days == days) & (eligible.units > 0)].sort_values()\n    if product_count not in (20, 50):\n        raise ValueError(\"Choose 20 or 50 products for the matched expansion experiment\")\n    if len(codes) < product_count:\n        raise ValueError(f\"Need {product_count} products with two complete training years and positive training sales\")\n    selected = sorted(np.random.default_rng(42).choice(codes.to_numpy(), 20, replace=False).tolist())\n    if product_count == 50:\n        remaining = codes[~codes.isin(selected)].to_numpy()\n        selected = sorted(selected + np.random.default_rng(42).choice(remaining, 30, replace=False).tolist())\n    history = data[data.product_id.isin(selected) & data.date.between(start, end)].copy()\n    expected_days = (end - start).days + 1\n    if not history.groupby(\"product_id\").size().eq(expected_days).all():\n        raise ValueError(\"Selected products have missing held-out days; do not silently resample\")\n    return history, {\n        \"source_url\": SOURCE_URL, \"source_revision\": SOURCE_REVISION,\n        \"store_id\": \"CA_1\", \"category\": \"FOODS\", \"seed\": 42,\n        \"products\": selected, \"eligible_products\": len(codes),\n        \"selection_rule\": \"Seeded sample of 20 training-eligible products; expansion retains those 20 and adds 30 seeded remaining products\" if product_count == 50 else \"Seeded uniform sample of sorted products complete and positive in training only\",\n        \"training_start\": str(start.date()), \"initial_training_end\": str(training_end.date()),\n        \"data_end\": str(end.date()), \"initial_training_days\": days,\n        \"observations\": len(history),\n        \"target\": \"Recorded sales units, not unconstrained demand\",\n    }\n\n\ndef sale_day_metrics(actual, predicted, threshold):\n    actual = np.asarray(actual) > 0\n    predicted = np.asarray(predicted) >= threshold\n    tp = int((actual & predicted).sum())\n    fp = int((~actual & predicted).sum())\n    fn = int((actual & ~predicted).sum())\n    tn = int((~actual & ~predicted).sum())\n    return {\n        \"tp\": tp, \"fp\": fp, \"fn\": fn, \"tn\": tn,\n        \"precision\": tp / (tp + fp) if tp + fp else None,\n        \"recall\": tp / (tp + fn) if tp + fn else None,\n        \"f1\": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,\n    }\n\n\ndef evaluate(history):\n    \"\"\"Frozen recipe: first two blocks validate tau; last block is the final test.\"\"\"\n    history = normalize_history(history)\n    training = build_training_rows(history, history.date.max())\n    predictions, folds = [], []\n    for number, (_, held_out) in enumerate(rolling_origin_split(history, \"date\", 28, 3), 1):\n        origin = held_out.date.min() - pd.Timedelta(days=1)\n        rows = training[training.origin_date + pd.Timedelta(days=28) <= origin]\n        if rows.empty:\n            raise ValueError(\"Insufficient training before holdout\")\n        model = _fit_model(rows)\n        features = build_feature_matrix(history, origin)\n        frame = features[[\"product_id\", \"target_date\", \"horizon_day\"]].copy()\n        frame[\"actual\"] = extract_training_targets(history, origin).to_numpy()\n        frame[\"predicted\"] = model.predict(features[FEATURE_COLUMNS])\n        frame[\"baseline\"] = np.concatenate([\n            seasonal_naive_forecast(group.units_sold.to_numpy(), 28)\n            for _, group in history[history.date <= origin].groupby(\"product_id\", sort=True)\n        ])\n        if not np.isfinite(frame[[\"actual\", \"predicted\", \"baseline\"]]).all().all():\n            raise ValueError(\"Nonfinite experiment values\")\n        frame[\"fold\"] = number\n        frame[\"split\"] = \"test\" if number == 3 else \"validation\"\n        predictions.append(frame)\n        folds.append({\"fold\": number, \"origin\": str(origin.date()),\n                      \"training_label_end\": str(rows.target_date.max().date()),\n                      \"training_rows\": len(rows), \"split\": frame.split.iloc[0]})\n        print(f\"Finished {frame.split.iloc[0]} fold {number}, origin {origin.date()}\", flush=True)\n    daily = pd.concat(predictions, ignore_index=True)\n    validation = daily[daily.split == \"validation\"]\n    thresholds = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]\n    threshold = max(thresholds, key=lambda tau: (\n        sale_day_metrics(validation.actual, validation.predicted, tau)[\"f1\"] or 0))\n    scores = []\n    for (fold, split), group in daily.groupby([\"fold\", \"split\"]):\n        for horizon in (7, 14, 28):\n            totals = group[group.horizon_day <= horizon].groupby(\"product_id\")[[\n                \"actual\", \"predicted\", \"baseline\"]].sum()\n            for pid, values in [(\"ALL\", totals), *[(pid, row.to_frame().T) for pid, row in totals.iterrows()]]:\n                for name in (\"predicted\", \"baseline\"):\n                    scores.append({\"fold\": int(fold), \"split\": split, \"horizon\": horizon,\n                                   \"product_id\": pid, \"model\": \"random_forest\" if name == \"predicted\" else \"seasonal_naive\",\n                                   **calculate_metrics(values.actual.to_numpy(), values[name].to_numpy())})\n    classification = {}\n    for split, group in daily.groupby(\"split\"):\n        classification[split] = {\n            \"random_forest\": sale_day_metrics(group.actual, group.predicted, threshold),\n            \"seasonal_naive\": sale_day_metrics(group.actual, group.baseline, 0.5),\n            \"always_sale\": sale_day_metrics(group.actual, np.ones(len(group)), 0.5),\n        }\n    return daily, pd.DataFrame(scores), {\"folds\": folds, \"sale_day_threshold\": threshold,\n                                         \"classification\": classification}\n\n\ndef write_discussion(scores, report, output):\n    lines = [\"# M5 forecasting experiment results\", \"\",\n             f\"Runtime: {report['runtime']}. Dataset revision: `{SOURCE_REVISION}`.\",\n             f\"Initial training: {report['training_start']}\u2013{report['initial_training_end']} \"\n             f\"({report['initial_training_days']} days), {len(report['products'])} CA_1 FOODS products.\",\n             \"First two 28-day blocks validate the sale-day threshold; the final 28 days are the test.\", \"\",\n             \"| Test horizon | RF MAE (units) | RF RMSE (units) | RF WAPE | Baseline WAPE |\",\n             \"|---|---:|---:|---:|---:|\"]\n    test = scores[(scores.split == \"test\") & (scores.product_id == \"ALL\")]\n    for h in (7, 14, 28):\n        group = test[test.horizon == h].set_index(\"model\")\n        rf, base = group.loc[\"random_forest\"], group.loc[\"seasonal_naive\"]\n        percent = lambda value: \"undefined\" if pd.isna(value) else f\"{100 * value:.2f}%\"\n        lines.append(f\"| {h} days | {rf.mae:.3f} | {rf.rmse:.3f} | {percent(rf.wape)} | {percent(base.wape)} |\")\n    lines += [\"\", \"MAE/RMSE above measure per-product horizon totals, not daily errors. \"\n              \"WAPE is an error fraction displayed as a percentage, not an accuracy score.\", \"\",\n              f\"Any-sale-day classification (threshold {report['sale_day_threshold']}, chosen on validation only):\"]\n    for name, metrics in report[\"classification\"][\"test\"].items():\n        lines.append(f\"- {name}: `{json.dumps(metrics)}`\")\n    per_product = scores[(scores.split == \"test\") & (scores.product_id != \"ALL\") & (scores.horizon == 28)]\n    comparison = per_product.pivot(index=\"product_id\", columns=\"model\", values=\"mae\")\n    losses = int((comparison.random_forest >= comparison.seasonal_naive).sum())\n    lines += [\"\", \"## Interpretation and weaknesses\", \"\",\n              f\"Random Forest fails to improve 28-day absolute error over baseline for {losses}/{len(report['products'])} products. \"\n              \"Inspect per-product scores; aggregate improvement can hide weak products.\",\n              \"High sale-day F1 can come from frequently positive sales. Compare with always_sale; \"\n              \"it does not establish quantity accuracy or stockout detection.\",\n              \"Only one final 28-day test window is available; two training years contain few annual cycles. \"\n              \"Walmart sales and US calendar effects do not establish local Store performance.\",\n              \"Recorded zeros do not identify stockouts or lost demand. The shared model currently uses \"\n              \"sales lags and calendar fields, not prices, promotions, or stock availability.\", \"\",\n              \"## Improvements to investigate\", \"\",\n              \"1. Inspect worst product/horizon errors and sustained over/underprediction in daily_predictions.csv.\",\n              \"2. Use validation to test price/promotion features only when known at the forecast origin; \"\n              \"do not use realized future prices or tune against this final test.\",\n              \"3. Evaluate additional future windows and locally collected Store history before operational approval.\", \"\",\n              \"## Integration boundary\", \"\",\n              \"Collect actual Store sales first. Train an isolated candidate when complete usable history \"\n              \"and chronological validation are sufficient; retain review/approval before daily forecast generation. \"\n              \"Public M5 history and models stay separate. Forecasts support replenishment and never change stock.\",\n              \"The existing readiness bands are below 56 days: Insufficient; 56\u2013179: Low; 180\u2013364: Medium; \"\n              \"365+: High only with acceptable validation. Minimum feature history is not sufficient \"\n              \"evidence for reliable validation or automatic approval.\",\n              \"No production activation or accuracy pass is inferred from this experiment.\"]\n    (output / \"discussion.md\").write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"parquet\", type=Path)\n    parser.add_argument(\"--output-dir\", type=Path, required=True)\n    parser.add_argument(\"--products\", type=int, choices=[20, 50], default=20)\n    parser.add_argument(\"--runtime\", choices=[\"local Python\", \"Google Colab\"], default=\"local Python\")\n    args = parser.parse_args()\n    if args.output_dir.exists():\n        parser.error(\"Output directory already exists; choose a new experiment directory\")\n    raw = pd.read_parquet(args.parquet, columns=[\"datetime\", \"item_id\", \"sales\", \"store_id\", \"cat_id\"],\n                          filters=[(\"store_id\", \"==\", \"CA_1\"), (\"cat_id\", \"==\", \"FOODS\")])\n    history, metadata = prepare_history(raw, args.products)\n    daily, scores, evaluation = evaluate(history)\n    report = {**metadata, **evaluation, \"feature_version\": FEATURE_VERSION,\n              \"parameters\": MODEL_PARAMETERS, \"features\": FEATURE_COLUMNS,\n              \"runtime\": args.runtime,\n              \"software\": {\"python\": platform.python_version(), \"numpy\": np.__version__,\n                           \"pandas\": pd.__version__, \"scikit_learn\": sklearn.__version__}}\n    with args.parquet.open(\"rb\") as stream:\n        report[\"source_sha256\"] = hashlib.file_digest(stream, \"sha256\").hexdigest()\n    args.output_dir.mkdir(parents=True, exist_ok=False)\n    history.to_csv(args.output_dir / \"daily_history.csv\", index=False)\n    daily.to_csv(args.output_dir / \"daily_predictions.csv\", index=False)\n    scores.to_csv(args.output_dir / \"scores.csv\", index=False)\n    (args.output_dir / \"metrics.json\").write_text(json.dumps(report, indent=2, allow_nan=False), encoding=\"utf-8\")\n    write_discussion(scores, report, args.output_dir)\n    print((args.output_dir / \"discussion.md\").read_text(encoding=\"utf-8\"))\n\n\nif __name__ == \"__main__\":\n    main()\n"
}

for name, source in BUNDLED_SOURCES.items():
    Path(name).write_text(source, encoding='utf-8')


In [ ]:
import sys
import urllib.request
sys.path.insert(0, str(work))
from m5_sales_experiment import SOURCE_URL
parquet = work / 'm5_daily.parquet'
if not parquet.exists():
    temporary = work / 'm5_daily.parquet.download'
    with urllib.request.urlopen(SOURCE_URL, timeout=120) as source, temporary.open('wb') as target:
        import shutil
        shutil.copyfileobj(source, target)
    temporary.replace(parquet)
print('Pinned source:', SOURCE_URL)
print('Downloaded bytes:', parquet.stat().st_size)


In [ ]:
import subprocess
PRODUCT_COUNT = 50  # Use 20 to reproduce the original comparison group.
from datetime import datetime, timezone
output = work / ('results-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
subprocess.run([sys.executable, str(work / 'm5_sales_experiment.py'), str(parquet),
                '--output-dir', str(output), '--products', str(PRODUCT_COUNT), '--runtime', 'Google Colab'], check=True)
from IPython.display import Markdown, display
display(Markdown((output / 'discussion.md').read_text(encoding='utf-8')))


## Read the results
- RF prediction: average of tree predictions. MAE = mean(abs(actual − predicted)); RMSE = sqrt(mean((actual − predicted)²)).
- Horizon scores compare each product's summed 7/14/28-day quantities. WAPE = sum(abs(error))/sum(actual), undefined with zero total actual. The CSV stores WAPE as a fraction; discussion displays a percentage. Lower quantity errors are better.
- Sale-day positive means actual units > 0; predicted positive means predicted units ≥ validation-selected threshold. Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = 2TP/(2TP+FP+FN). Undefined metrics are null. F1 and F1 score mean the same thing.
- Compare classification against **always_sale**, and quantity errors against **seasonal_naive** (last week repeated). High F1 is not quantity accuracy or stockout detection.
- Inspect all products, not just favorable examples. A high RMSE relative to MAE suggests large misses. Confirm weaknesses in per-product/day errors before choosing improvements. No operational pass criterion has been agreed.

[Metric definitions](https://scikit-learn.org/stable/modules/model_evaluation.html)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
predictions = pd.read_csv(output / 'daily_predictions.csv', parse_dates=['target_date'])
scores = pd.read_csv(output / 'scores.csv')
display(scores[(scores.split == 'test') & (scores.product_id == 'ALL')])
test = predictions[predictions.split == 'test']
fig, axes = plt.subplots((test.product_id.nunique() + 3) // 4, 4, figsize=(18, 3.2 * ((test.product_id.nunique() + 3) // 4)), sharex=True)
for ax, (pid, group) in zip(axes.flat, test.groupby('product_id')):
    for column in ['actual', 'predicted', 'baseline']:
        ax.plot(group.target_date, group[column], label=column)
    ax.set_title(pid)
    ax.set_ylabel('Units/day')
    ax.tick_params(axis='x', rotation=45)
for ax in axes.flat[test.product_id.nunique():]:
    ax.set_visible(False)
axes.flat[0].legend()
fig.suptitle('Final held-out test: every selected product')
fig.tight_layout()
fig.savefig(output / 'test_predictions.png', dpi=140)
plt.show()


## Save results before the runtime ends
The ZIP contains provenance, selected products/dates, package versions, metrics, per-product scores, held-out daily predictions, history, charts and generated discussion. It contains no production model or Store records. Bring `metrics.json`, `scores.csv` and `discussion.md` back for review before integration. [Colab runtime limitations](https://research.google.com/colaboratory/faq.html).

Later integration: collect Store sales → validate usable history → queued candidate training → chronological evaluation and approval → forecasts after close → advisory replenishment. Existing readiness starts at 56 days (Low), improves at 180 days (Medium), and reaches High at 365+ only with acceptable validation. Having enough rows to fit does not guarantee sufficient history for the full validation protocol.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(output), 'zip', output)
files.download(archive)
